# 05 路口通行计数审计

某路口影像分析已经生成轨迹。你的任务是审查交通量提取规则：设置虚拟线、指定方向、限制跨帧间隔并检查重复计数。交付可逐条回看的事件清单。

建议工作量：8—12小时。本工作本是项目起点，默认程序的输出不是完整作业答案。

## 最终成果
- 计数线与方向定义
- 带车辆ID和时间的跨线事件CSV
- 采样与断点敏感性分析
- 人工核查方案及误差来源说明


## 数据与范围

[SinD 官方仓库与样本格式](https://github.com/SOTIF-AVLab/SinD)

数据采用禁止商业使用的自定义条款，保留原LICENSE，不称为标准CC0

固定官方提交的天津8_2_1车辆平滑轨迹前120秒。保留原ID、时间与米制坐标，行人文件不包含在此样本中。

- 回放的是公开平滑轨迹，不是原始视频或本课程检测器输出。
- 本项目不声称完成YOLO训练或检验完整视觉系统精度。
- 跨线数量对线的位置、方向、采样和轨迹断点敏感。
- 轨迹ID可能已由数据提供方处理，仍需独立人工标注来验证真实计数。


In [ ]:
from pathlib import Path
import sys, json
candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p / "python" / "analyze.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the project-kit folder or its notebooks folder")
sys.path.insert(0, str(ROOT / "python"))
from analyze import run
OUTPUT = ROOT / "outputs" / "student"
OUTPUT.mkdir(parents=True, exist_ok=True)


## 参考分析与口径检查

先运行一次，解释每个指标的分母和单位。打开源码确认数据筛选规则。预测项目这里读取冻结模型的结果，不重新训练。


In [ ]:
project = "counting"
config = {
  "line": "15",
  "direction": "both",
  "stride": "1",
  "band": "0.3",
  "gap": "1.5",
  "type": "all"
}
result = run(project, config, ROOT / "data")
print(json.dumps(result["metrics"], ensure_ascii=False, indent=2))
print("Source SHA-256:", result["sourceSha"])


## 对照实验

以下配置提供一个可运行起点。说明每次只改变了什么，以及还存在哪些混杂条件。增加你自己的对照，不只重复默认结果。


In [ ]:
comparisons = [
  {
    "stride": "1",
    "gap": "1.5"
  },
  {
    "stride": "10",
    "gap": "1.5"
  },
  {
    "stride": "10",
    "gap": "0.3"
  }
]
experiments = []
for change in comparisons:
    trial = run(project, {**config, **change}, ROOT / "data")
    experiments.append({"project": project, "config": trial["config"], "metrics": trial["metrics"], "sourceSha": trial["sourceSha"]})
    print(json.dumps(experiments[-1], ensure_ascii=False))
(OUTPUT / (project + "-comparison.json")).write_text(json.dumps(experiments, ensure_ascii=False, indent=2), encoding="utf-8")


## 01 计数定义

**明确一次通行事件**

写明x方向虚拟线、正负方向、车辆类别和去重规则。为什么检测框总数不等于通过车辆数？

阶段成果：可执行的事件定义及适用场景。

### 我的证据与解释

在这里填写自己的分析，引用结果行、实验参数或图表。


## 02 轨迹核查

**回看真实记录**

拖动时间，检查轨迹与计数线关系。选3个ID核对事件清单，不把回放称作原视频。

阶段成果：3个ID的时间与方向核查记录。

### 我的证据与解释

在这里填写自己的分析，引用结果行、实验参数或图表。


## 03 规则对照

**检验采样与断点**

改变采样间隔、死区宽度、最大断点和方向，保存至少3次结果，解释差异。

阶段成果：至少3组配置和事件数对照，包括漏计或重复风险。

### 我的证据与解释

在这里填写自己的分析，引用结果行、实验参数或图表。


## 04 系统边界

**设计独立核验**

若要评价完整视频系统，还需要哪些人工标签、原视频、检测与跟踪指标？

阶段成果：人工核查方案，区分后处理敏感性与真实计数准确率。

### 我的证据与解释

在这里填写自己的分析，引用结果行、实验参数或图表。


## 深入分析

- 在自己获授权的视频上运行检测跟踪，将结果转换为同一事件接口。
- 增加有限长度计数线与多入口转向统计，补充遮挡及ID切换测试。

项目包还包含 extensions.py 的可运行扩展。先安装 python/requirements.txt 中的依赖，再运行下面的命令。


In [ ]:
import subprocess
subprocess.run([sys.executable, str(ROOT / "python" / "extensions.py"), "--project", "counting", "--output", str(OUTPUT / "extensions")], check=True)


## 提交前自查

- [ ] 可导出逐条跨线事件，不只给最终数字。
- [ ] 参数变化可复现，长断点不盲目连接。
- [ ] 没有用轨迹回放宣称检测准确率，保留原数据许可。

报告应包含研究问题、方法对照、发现、局限、源数据哈希和复现命令。请附代码、配置、结果CSV。阶段文字与实验次数不自动换算成绩。
